# Final Model Evaluation and Comparison


## Introduction

At this stage, all candidate models developed throughout the project are evaluated under a common framework using the hold-out test set. The objective is to compare predictive performance across model families in a consistent and reproducible way.

Rather than relying on results reported separately in earlier notebooks, each saved model artifact is loaded and applied to the same test data using its corresponding preprocessing configuration. This ensures that all models are evaluated under identical conditions and that the comparison reflects true differences in modeling approach rather than differences in data preparation or evaluation procedure.

The comparison focuses primarily on PR-AUC, which remains the most informative metric in the presence of class imbalance and aligns with the objective of identifying customers at risk of churn. Additional metrics, including ROC-AUC, accuracy, precision, recall, and F1 score, are also reported to provide a broader view of model behavior.

This chapter serves as the final comparative evaluation stage before selecting the most appropriate model for deployment.

## Table of Contents

1. [Load Data](#modeling-considerations)
2. [Load Model Artifacts](#load-model-artifacts)
4. [Prepare Data for Models](#prepare-data-for-models)
3. [Get Results for Each Model](#get-results-for-each-model)
4. [Threshold Analysis](#threshold-analysis)
5. [Final Model Selection](#final-model-selection)
6. [Executvie summary](#executive-summary)


## Load Data

The final comparative evaluation is carried out on the hold-out test set, which was not used during model training or hyperparameter optimization. While intermediate model notebooks included a one-time evaluation on the test data for inspection purposes, the test set was never used to influence model selection, feature engineering decisions, or tuning. Its role in this chapter is therefore to provide a consistent and unbiased reference point for comparing all finalized models.

Although no further model fitting is performed, the raw test data cannot be used directly. To ensure full reproducibility, the same configuration-driven preparation pipeline developed in earlier stages of the project is reapplied. Starting from the saved raw test dataset, the sequence of preprocessing and feature-engineering artifacts is loaded and executed in the original order, producing a standardized prepared test dataset.

This prepared dataset then serves as the common input source for all saved model artifacts. Each model-specific artifact reconstructs its expected feature representation from this shared prepared data, including any required column alignment or scaling. In this way, all final models are evaluated on the same unseen data under identical and fully reproducible conditions.

In [1]:
from src.utils.data_preparation_pipeline import apply_artifact
import polars as pl
import json

test_df_raw = pl.read_csv("./data/processed/02_test_dataset.csv")

with open("./data/configs/01_eda.json") as f:
    cfg_01 = json.load(f)

with open("./data/configs/02_ram.json") as f:
    cfg_02 = json.load(f)

with open("./data/configs/03_fe.json") as f:
    cfg_03 = json.load(f)

with open("./data/configs/04_fe.json") as f:
    cfg_04 = json.load(f)

with open("./data/configs/05_fe.json") as f:
    cfg_05 = json.load(f)

with open("./data/configs/06_fe.json") as f:
    cfg_06 = json.load(f)

artifacts = [cfg_01, cfg_02, cfg_03, cfg_04, cfg_05, cfg_06]

df_out = test_df_raw
for artifact in artifacts:
    df_out = apply_artifact(df_out, artifact)
test_df_prepared = df_out
test_df_prepared.head()

Partner,Dependents,tenure,InternetService,Contract,PaperlessBilling,MonthlyCharges,Churn,SeniorCitizenRelevel,AdditionalInternetServicesCount,StreamingServicesCount,MC_x_Additional,MC_x_Streaming,PaymentMethod_bin_3
str,str,i64,str,str,str,f64,str,str,f64,f64,f64,f64,str
"""Yes""","""No""",42,"""Fiber optic""","""Month-to-month""","""Yes""",84.3,"""Yes""","""No""",2.0,0.0,168.6,0.0,"""Electronic check"""
"""No""","""No""",7,"""Fiber optic""","""Month-to-month""","""Yes""",79.0,"""Yes""","""No""",0.0,1.0,0.0,79.0,"""Electronic check"""
"""No""","""No""",1,"""Fiber optic""","""Month-to-month""","""Yes""",70.25,"""Yes""","""No""",0.0,0.0,0.0,0.0,"""Electronic check"""
"""No""","""No""",10,"""Fiber optic""","""Month-to-month""","""Yes""",84.6,"""Yes""","""Yes""",1.0,1.0,84.6,84.6,"""Automatic"""
"""Yes""","""Yes""",17,"""No""","""Month-to-month""","""No""",24.1,"""Yes""","""No""",0.0,0.0,0.0,0.0,"""Mailed check"""


In [2]:
target_col = "Churn"

print("Target column present:", target_col in test_df_prepared.columns)

Target column present: True


## Load Model Artifacts

All models developed throughout the project have been saved together with their preprocessing configuration and hyperparameters. These saved artifacts enable the models to be reloaded and evaluated without retraining, ensuring full reproducibility of results.

Each artifact contains not only the trained model itself, but also the metadata required to reconstruct the exact feature representation used during training. This includes definitions of categorical and numerical variables, category ordering, reference column structure, and, where applicable, fitted scaling transformations. As a result, each model can be applied to the test data in precisely the same form in which it was originally trained.

In this section, all model artifacts are loaded from disk and organized into a unified dictionary structure. This structure maps model identifiers to their corresponding artifacts and serves as the central interface for the evaluation process.

Before loading the saved neural network artifacts, the `ChurnNN` class must be defined in the current environment. This is necessary because the neural network models were serialized as full Python objects. During deserialization, pickle requires access to the original class definition in order to correctly reconstruct the model.

In [3]:
from src.utils.neural_networks import ChurnNN

In [4]:
import pickle
import os

models_dir = "./models"

model_files = [
    "logistic_regression_final.pkl",
    "decision_tree_final.pkl",
    "random_forest_final.pkl",
    "hist_boosted_trees_final.pkl",
    "xgboost_final.pkl",
    "lightgbm_final.pkl",
    "linear_svm_final.pkl",
    "rbf_svm_final.pkl",
    "nn_16.pkl",
    "nn_32.pkl",
    "nn_64.pkl"
]

artifacts = {}

for file in model_files:
    path = os.path.join(models_dir, file)
    
    with open(path, "rb") as f:
        artifact = pickle.load(f)
    
    model_name = file.replace(".pkl", "")
    artifacts[model_name] = artifact

print(f"Loaded {len(artifacts)} model artifacts.")
list(artifacts.keys())

Loaded 11 model artifacts.


['logistic_regression_final',
 'decision_tree_final',
 'random_forest_final',
 'hist_boosted_trees_final',
 'xgboost_final',
 'lightgbm_final',
 'linear_svm_final',
 'rbf_svm_final',
 'nn_16',
 'nn_32',
 'nn_64']

With all model artifacts successfully loaded, the next step is to prepare the test data in the format required by each model.

## Prepare Data for Models

With both the prepared test dataset and the model artifacts available, the next step is to construct the model-specific input data required for evaluation.

Although all models are evaluated on the same underlying test dataset, they were trained using different feature representations. These differences include encoding strategies, feature subsets, column ordering, and, in some cases, scaling transformations. Therefore, the prepared test data must be transformed separately for each model to match its original training configuration.

This transformation is performed using the preprocessing information stored within each artifact. For each model, the corresponding artifact is passed together with the prepared test dataset into a dedicated data loader, which reconstructs the exact feature matrix and target vector used during training.

As a result, each model receives input data in the precise format it expects, ensuring consistency between training and evaluation and enabling a fair comparison across all models.

In [5]:
from src.model_data_loader.load_data import prepare_data_from_artifact

# Logistic Regression
X_test_lr, y_test_lr = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["logistic_regression_final"],
    model_type="logistic_regression"
)

# Tree-based models
X_test_dt, y_test_dt = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["decision_tree_final"],
    model_type="tree_model"
)

X_test_rf, y_test_rf = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["random_forest_final"],
    model_type="tree_model"
)

X_test_hgb, y_test_hgb = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["hist_boosted_trees_final"],
    model_type="tree_model"
)

X_test_xgb, y_test_xgb = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["xgboost_final"],
    model_type="tree_model"
)

X_test_lgbm, y_test_lgbm = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["lightgbm_final"],
    model_type="tree_model"
)

# SVM models
X_test_svm_lin, y_test_svm_lin = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["linear_svm_final"],
    model_type="svm"
)

X_test_svm_rbf, y_test_svm_rbf = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["rbf_svm_final"],
    model_type="svm"
)

# Neural Networks
X_test_nn_16, y_test_nn_16 = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["nn_16"],
    model_type="nn"
)

X_test_nn_32, y_test_nn_32 = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["nn_32"],
    model_type="nn"
)

X_test_nn_64, y_test_nn_64 = prepare_data_from_artifact(
    df=test_df_prepared,
    artifact=artifacts["nn_64"],
    model_type="nn"
)

With the model-specific test datasets prepared, each model can now be applied to the test data to generate predictions and evaluate performance.

## Get Results for Each Model

With the model-specific test datasets prepared, each trained model can now be applied to generate predictions and evaluate performance.

For each model, predicted probabilities are first obtained using the appropriate prediction interface. These probabilities are then converted into class predictions using a fixed threshold of 0.5. Using a common threshold across all models ensures a consistent and unbiased baseline comparison, independent of any model-specific threshold tuning performed during development.

Performance is evaluated using a unified set of classification metrics, including accuracy, precision, recall, F1-score, ROC-AUC, and PR-AUC. These metrics capture both overall predictive performance and the trade-offs between correctly identifying positive cases and avoiding false positives.

All results are collected into a single comparison table, where each row corresponds to one model and its associated performance metrics. This table provides a direct and standardized basis for comparing all modeling approaches under identical conditions.

In [6]:
import torch
import pandas as pd

from src.utils.classification_metrics import compute_classification_metrics

# -------------------------------------------------------------------
# Collect prepared test sets
# -------------------------------------------------------------------
test_sets = {
    "logistic_regression_final": (X_test_lr, y_test_lr),
    "decision_tree_final": (X_test_dt, y_test_dt),
    "random_forest_final": (X_test_rf, y_test_rf),
    "hist_boosted_trees_final": (X_test_hgb, y_test_hgb),
    "xgboost_final": (X_test_xgb, y_test_xgb),
    "lightgbm_final": (X_test_lgbm, y_test_lgbm),
    "linear_svm_final": (X_test_svm_lin, y_test_svm_lin),
    "rbf_svm_final": (X_test_svm_rbf, y_test_svm_rbf),
    "nn_16": (X_test_nn_16, y_test_nn_16),
    "nn_32": (X_test_nn_32, y_test_nn_32),
    "nn_64": (X_test_nn_64, y_test_nn_64),
}

# -------------------------------------------------------------------
# Score each model on the test set
# -------------------------------------------------------------------
results = []

for model_name, artifact in artifacts.items():
    model = artifact["model"]
    X_test, y_test = test_sets[model_name]

    # Convert target to numpy if needed
    y_true = y_test.values if hasattr(y_test, "values") else y_test

    # Statsmodels logistic regression
    if model_name == "logistic_regression_final":
        y_prob = model.predict(X_test)
        threshold = 0.5
        y_pred = (y_prob >= threshold).astype(int)

    # PyTorch neural networks
    elif artifact.get("model_type") == "pytorch_nn":
        model.eval()

        X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

        with torch.no_grad():
            y_prob = model(X_test_tensor).detach().cpu().numpy().reshape(-1)

        threshold = artifact.get("threshold", 0.5)
        y_pred = (y_prob >= threshold).astype(int)

    # sklearn models with predict_proba
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
        threshold = artifact.get("threshold", 0.5)
        y_pred = (y_prob >= threshold).astype(int)

    metrics = compute_classification_metrics(
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob
    )

    results.append({
        "model": model_name,
        "threshold": threshold,
        **metrics
    })

# -------------------------------------------------------------------
# Final comparison table
# -------------------------------------------------------------------
results_df = pd.DataFrame(results).sort_values(
    by=["pr_auc", "auc"],
    ascending=False
).reset_index(drop=True)

## Model Performance Comparison and Interpretation

In [7]:
results_df

,model,threshold,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,hist_boosted_trees_final,0.5,0.792761,0.645390,0.486631,0.554878,0.848383,0.686113,182,935,100,192
1,random_forest_final,0.5,0.791341,0.648148,0.467914,0.543478,0.845250,0.677469,175,940,95,199
2,nn_32,0.5,0.794180,0.631250,0.540107,0.582133,0.836637,0.659241,202,917,118,172
3,rbf_svm_final,0.5,0.786373,0.643137,0.438503,0.521463,0.832818,0.658481,164,944,91,210
4,logistic_regression_final,0.5,0.789212,0.631399,0.494652,0.554723,0.834207,0.655219,185,927,108,189
5,nn_16,0.5,0.786373,0.607670,0.550802,0.577840,0.834364,0.654900,206,902,133,168
6,linear_svm_final,0.5,0.734564,0.000000,0.000000,0.000000,0.830784,0.652776,0,1035,0,374
7,nn_64,0.5,0.788502,0.612426,0.553476,0.581461,0.833563,0.649906,207,904,131,167
8,lightgbm_final,0.5,0.787083,0.614198,0.532086,0.570201,0.827451,0.648004,199,910,125,175
9,xgboost_final,0.5,0.786373,0.615142,0.521390,0.564399,0.821933,0.636898,195,913,122,179


The comparison of all models on the hold-out test set reveals several important patterns regarding predictive performance, model behavior, and trade-offs between precision and recall.

### Overall Performance

Across all evaluated models, performance is relatively close, indicating that the feature engineering pipeline provides a strong and stable signal regardless of the modeling approach. Most models achieve accuracy around 0.78–0.79 and ROC-AUC values in the range of approximately 0.82–0.85, suggesting consistently good discrimination between churn and non-churn customers.

Among all models, the **histogram-based gradient boosting model** achieves the highest ROC-AUC (0.848) and PR-AUC (0.686), indicating the strongest overall ranking performance. This suggests that boosted tree methods are particularly effective at capturing non-linear relationships and interactions in the data.

### Precision–Recall Trade-off

Despite similar overall metrics, the models exhibit meaningful differences in how they balance precision and recall:

- Tree-based ensemble models (Random Forest, Gradient Boosting) tend to favor **higher precision** but slightly lower recall.
- Neural networks and SVMs generally achieve **higher recall**, at the cost of more false positives.
- Logistic regression remains competitive, providing a balanced trade-off with stable and interpretable performance.

For example:
- The **Random Forest** achieves one of the highest precision values (0.648), but with lower recall (0.468), indicating a more conservative prediction strategy.
- In contrast, **neural networks (nn_16, nn_64)** achieve higher recall (≈0.55), identifying more churn cases but at the cost of increased false positives.

This highlights an important practical consideration:  
different models may be preferable depending on whether the priority is minimizing false positives or maximizing churn detection.

### Neural Networks vs Tree-Based Models

Neural networks (particularly `nn_32` and `nn_64`) perform competitively with tree-based models, achieving the highest F1-scores (≈0.58) and the highest recall among all models. However, their ROC-AUC and PR-AUC are slightly lower than those of gradient boosting models.

This suggests that while neural networks are effective at identifying positive cases, they are slightly less well-calibrated in ranking observations compared to boosted trees.

### Simpler Models Remain Competitive

Interestingly, simpler models such as **logistic regression** and **linear SVM** perform only marginally worse than more complex models. Logistic regression achieves an ROC-AUC of 0.834 and an F1-score comparable to several more advanced models.

This indicates that:
- the engineered features capture a large portion of the signal
- model complexity provides incremental, but not dramatic, improvements

### Key Takeaways

- All models perform within a relatively narrow range, confirming the robustness of the feature engineering pipeline.
- **Histogram-based gradient boosting** provides the best overall ranking performance.
- **Neural networks** achieve the highest recall and F1-scores, making them strong candidates when detecting as many churn cases as possible is the priority.
- **Tree-based ensembles** offer a more conservative balance with higher precision.
- Simpler linear models remain competitive and may be preferred when interpretability is important.

It is important to note that all models were evaluated using a fixed classification threshold of 0.5. In practice, this threshold can be adjusted to better align with business objectives, and further analysis of threshold selection is presented in the next section.

## Threshold Analysis

The previous comparison evaluated all models using a fixed classification threshold of 0.5, providing a consistent baseline for performance comparison. However, in real-world applications, the choice of classification threshold plays a crucial role in determining how model predictions are translated into decisions.

Different thresholds lead to different trade-offs between precision and recall. A lower threshold increases the number of predicted positive cases, improving recall but potentially introducing more false positives. Conversely, a higher threshold results in more conservative predictions, improving precision at the cost of missed positive cases.

To better understand these trade-offs, a subset of representative models is selected for further analysis. Specifically, the following models are considered:

- **Histogram-based Gradient Boosting** (`hist_boosted_trees_final`), representing the strongest overall ranking performance  
- **Random Forest** (`random_forest_final`), representing a more conservative, high-precision model  
- **Neural Network (32 units)** (`nn_32`), representing a model with higher recall and balanced performance  
- **Logistic Regression** (`logistic_regression_final`), serving as a baseline and interpretable reference model  

For each of these models, performance is evaluated across a range of classification thresholds. This analysis follows the same approach previously applied to logistic regression, where model predictions are assessed over a grid of threshold values to examine how key metrics evolve.

This allows us to identify thresholds that maximize specific metrics (such as F1-score) or align better with different operational objectives, providing a more nuanced and decision-oriented comparison of model performance.

In [8]:
import torch
import pandas as pd

from src.utils.classification_metrics import evaluate_thresholds


# Selected models for threshold analysis
selected_models = {
    "logistic_regression_final": (artifacts["logistic_regression_final"], X_test_lr, y_test_lr),
    "hist_boosted_trees_final": (artifacts["hist_boosted_trees_final"], X_test_hgb, y_test_hgb),
    "random_forest_final": (artifacts["random_forest_final"], X_test_rf, y_test_rf),
    "nn_32": (artifacts["nn_32"], X_test_nn_32, y_test_nn_32),
}

# Evaluate thresholds for each selected model
threshold_results = {}

for model_name, (artifact, X_test, y_test) in selected_models.items():
    model = artifact["model"]
    y_true = y_test.values if hasattr(y_test, "values") else y_test

    # Logistic regression (statsmodels)
    if model_name == "logistic_regression_final":
        y_prob = model.predict(X_test)

    # Neural network (PyTorch)
    elif artifact.get("model_type") == "pytorch_nn":
        model.eval()

        X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

        with torch.no_grad():
            y_prob = model(X_test_tensor).detach().cpu().numpy().reshape(-1)

    # Tree-based sklearn models
    else:
        y_prob = model.predict_proba(X_test)[:, 1]

    threshold_df = evaluate_thresholds(
        y_true=y_true,
        y_prob=y_prob
    )

    threshold_df["model"] = model_name
    threshold_results[model_name] = threshold_df


# Combine all threshold results into one table
threshold_results_df = pd.concat(
    threshold_results.values(),
    axis=0,
    ignore_index=True
)

In [9]:
threshold_results_df.head()

,threshold,accuracy,precision,recall,f1,tp,tn,fp,fn,auc,pr_auc,model
0,0.01,0.308730,0.277448,1.000000,0.434379,374,61,974,0,0.834207,0.655219,logistic_regression_final
1,0.02,0.391767,0.303499,0.997326,0.465377,373,179,856,1,0.834207,0.655219,logistic_regression_final
2,0.03,0.442867,0.320524,0.981283,0.483213,367,257,778,7,0.834207,0.655219,logistic_regression_final
3,0.04,0.486160,0.337963,0.975936,0.502063,365,320,715,9,0.834207,0.655219,logistic_regression_final
4,0.05,0.513840,0.349176,0.962567,0.512456,360,364,671,14,0.834207,0.655219,logistic_regression_final


### Interpretation of Threshold Metrics

The evaluation across different classification thresholds shows how performance metrics such as precision, recall, and F1-score vary depending on the chosen decision boundary.

As expected, lowering the threshold increases recall at the cost of precision, while increasing the threshold leads to more conservative predictions with higher precision but lower recall. This behavior reflects the fundamental trade-off between identifying positive cases and avoiding false positives.

It is important to note that **ROC-AUC and PR-AUC remain constant across all threshold values**. This is because these metrics are computed based on the full range of predicted probabilities rather than a single classification threshold. They evaluate the model's ability to rank observations correctly, independent of how those probabilities are converted into class labels.

In contrast, metrics such as accuracy, precision, recall, and F1-score depend directly on the chosen threshold, as they are calculated from discrete class predictions. Therefore, threshold selection plays a critical role in determining the practical performance of a model in a given application.

In [10]:
best_thresholds_f1 = (
    threshold_results_df
    .sort_values(["model", "f1"], ascending=[True, False])
    .groupby("model")
    .first()
    .reset_index()
)

best_thresholds_f1[[
    "model", "threshold", "f1", "precision", "recall", "accuracy"
]]

,model,threshold,f1,precision,recall,accuracy
0,hist_boosted_trees_final,0.34,0.646635,0.587336,0.719251,0.791341
1,logistic_regression_final,0.28,0.630650,0.536585,0.764706,0.762243
2,nn_32,0.28,0.622363,0.513937,0.788770,0.745919
3,random_forest_final,0.34,0.645238,0.581545,0.724599,0.788502


### Threshold Stability and Global Decision Rule

The optimal thresholds identified for individual models fall within a relatively narrow range (approximately 0.28 to 0.34). This indicates that the decision boundary separating churn and non-churn customers is stable across different modeling approaches.

To assess whether a unified decision rule can be applied, all models were evaluated using a common threshold of 0.31, corresponding to the value previously identified for logistic regression.

The results show that model performance remains largely consistent under this shared threshold, with only minor variations in F1-score and precision-recall trade-offs. This suggests that a single global threshold can be used without significantly degrading performance.

From a practical perspective, this is an important finding. Using a unified threshold simplifies deployment and ensures consistent decision-making across models, while still maintaining strong predictive performance.

## Re-evaluation with a Global Threshold

The previous analysis identified model-specific optimal thresholds that maximize performance metrics such as F1-score. While this approach provides the best possible performance for each individual model, it does not reflect how models are typically deployed in practice.

In real-world applications, it is often desirable to use a single, consistent decision threshold across models to ensure interpretability, stability, and simplicity of the decision-making process. This is particularly relevant in operational settings, where maintaining multiple model-specific thresholds can increase system complexity and reduce transparency.

The analysis of optimal thresholds revealed that the best-performing values for different models lie within a relatively narrow range. This suggests that the underlying decision boundary is stable across modeling approaches.

To evaluate this observation, all models are re-evaluated using a common global threshold. This allows for a direct comparison under a unified decision rule and provides insight into how much performance is affected when moving from model-specific optimization to a standardized threshold.

In [11]:
global_threshold = 0.31

global_results = []

for model_name, artifact in artifacts.items():
    model = artifact["model"]
    X_test, y_test = test_sets[model_name]

    y_true = y_test.values if hasattr(y_test, "values") else y_test

    # logistic
    if model_name == "logistic_regression_final":
        y_prob = model.predict(X_test)

    # nn
    elif artifact.get("model_type") == "pytorch_nn":
        model.eval()
        X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

        with torch.no_grad():
            y_prob = model(X_test_tensor).detach().cpu().numpy().reshape(-1)

    # sklearn
    else:
        y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob >= global_threshold).astype(int)

    metrics = compute_classification_metrics(
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob
    )

    global_results.append({
        "model": model_name,
        "threshold": global_threshold,
        **metrics
    })

global_results_df = pd.DataFrame(global_results).sort_values(
    by="f1", ascending=False
)

global_results_df

,model,threshold,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
2,random_forest_final,0.31,0.775018,0.555126,0.767380,0.644220,0.845250,0.677469,287,805,230,87
3,hist_boosted_trees_final,0.31,0.772179,0.551859,0.754011,0.637288,0.848383,0.686113,282,806,229,92
10,nn_64,0.31,0.739532,0.505882,0.804813,0.621259,0.833563,0.649906,301,741,294,73
9,nn_32,0.31,0.756565,0.529190,0.751337,0.620994,0.836637,0.659241,281,785,250,93
0,logistic_regression_final,0.31,0.767211,0.546939,0.716578,0.620370,0.834207,0.655219,268,813,222,106
1,decision_tree_final,0.31,0.765082,0.543260,0.721925,0.619977,0.830691,0.629696,270,808,227,104
8,nn_16,0.31,0.750177,0.520295,0.754011,0.615721,0.834364,0.654900,282,775,260,92
5,lightgbm_final,0.31,0.770050,0.553879,0.687166,0.613365,0.827451,0.648004,257,828,207,117
4,xgboost_final,0.31,0.759404,0.536534,0.687166,0.602579,0.821933,0.636898,257,813,222,117
7,rbf_svm_final,0.31,0.702626,0.466765,0.844920,0.601332,0.832818,0.658481,316,674,361,58


Original results with threshold of 0.5, sorted by F1 measure:

In [12]:
results_df.sort_values(
    by="f1", ascending=False
)

,model,threshold,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
2,nn_32,0.5,0.794180,0.631250,0.540107,0.582133,0.836637,0.659241,202,917,118,172
7,nn_64,0.5,0.788502,0.612426,0.553476,0.581461,0.833563,0.649906,207,904,131,167
5,nn_16,0.5,0.786373,0.607670,0.550802,0.577840,0.834364,0.654900,206,902,133,168
8,lightgbm_final,0.5,0.787083,0.614198,0.532086,0.570201,0.827451,0.648004,199,910,125,175
9,xgboost_final,0.5,0.786373,0.615142,0.521390,0.564399,0.821933,0.636898,195,913,122,179
10,decision_tree_final,0.5,0.788502,0.625828,0.505348,0.559172,0.830691,0.629696,189,922,113,185
0,hist_boosted_trees_final,0.5,0.792761,0.645390,0.486631,0.554878,0.848383,0.686113,182,935,100,192
4,logistic_regression_final,0.5,0.789212,0.631399,0.494652,0.554723,0.834207,0.655219,185,927,108,189
1,random_forest_final,0.5,0.791341,0.648148,0.467914,0.543478,0.845250,0.677469,175,940,95,199
3,rbf_svm_final,0.5,0.786373,0.643137,0.438503,0.521463,0.832818,0.658481,164,944,91,210


## Impact of Global Threshold on Model Ranking

Re-evaluating all models using a common threshold of 0.31 leads to a notable shift in model performance and ranking based on F1 measure compared to the baseline evaluation at threshold 0.5.

### Shift in Model Ranking

At the default threshold of 0.5, neural networks achieved the highest F1-scores, primarily due to their relatively balanced precision and recall. However, when applying the global threshold of 0.31, tree-based ensemble methods—particularly **Random Forest** and **Histogram-based Gradient Boosting**—emerge as the top-performing models in terms of F1-score.

This shift highlights that:
- Neural networks performed well under the default threshold due to their inherent calibration around 0.5  
- Tree-based models, while initially more conservative, benefit significantly from a lower threshold, which unlocks higher recall without excessive loss in precision  

### Recall–Precision Trade-off

Lowering the threshold leads to a consistent increase in recall across all models, as more observations are classified as positive. This comes at the cost of increased false positives, reflected in reduced precision.

For example:
- **Random Forest** recall increases from ~0.47 to ~0.77  
- **Histogram Gradient Boosting** recall increases from ~0.49 to ~0.75  

Despite this shift, both models maintain a reasonable level of precision (~0.55), resulting in a substantial improvement in F1-score.

Neural networks, which already exhibited higher recall at threshold 0.5, gain less from threshold adjustment and instead experience a more noticeable drop in precision due to a larger number of false positives.

### Stability of Model Performance

Although model rankings change, overall performance remains relatively close across models. The top-performing models at the global threshold differ only marginally in F1-score, suggesting that the choice of model is less critical than the choice of threshold.

This reinforces an important insight:


**Threshold selection has a comparable, if not greater, impact on performance than the choice of model itself.**

### Practical Implications

From a deployment perspective, the use of a global threshold simplifies decision-making and ensures consistency across models. The observed stability of optimal thresholds (within the range of approximately 0.28 to 0.34) supports the use of such a unified threshold without significant loss in performance.

Under this unified decision rule:
- **Random Forest** and **Histogram Gradient Boosting** provide the strongest overall performance  
- **Neural networks** remain competitive, particularly when recall is prioritized  
- **Logistic regression** continues to offer a robust and interpretable baseline  

### Key Takeaway

The analysis demonstrates that model evaluation should not rely solely on a fixed default threshold. Instead, understanding how model performance changes across thresholds is essential for making informed decisions.

Ultimately, the results show that:
- model selection and threshold selection are tightly interconnected  
- a well-chosen threshold can significantly alter model ranking  
- and a global threshold can provide a practical and effective compromise for real-world deployment

## Final Model Selection

Based on the comprehensive evaluation across multiple models and threshold configurations, the **Histogram-based Gradient Boosting model (`hist_boosted_trees_final`)** is selected as the final model.

The primary objective of this project is not only accurate classification, but also the ability to effectively **rank customers by their likelihood of churn**. In this context, metrics such as ROC-AUC and PR-AUC are of particular importance, as they evaluate the model's ability to correctly order observations based on predicted probabilities.

Among all evaluated models, the histogram-based gradient boosting model consistently achieved the highest ROC-AUC and PR-AUC scores. This indicates superior ranking performance and suggests that the model captures complex non-linear relationships in the data more effectively than alternative approaches.

In addition to its strong ranking capability, the model also demonstrates competitive performance in threshold-dependent metrics. When evaluated using the global threshold of 0.31, it achieves one of the highest F1-scores among all models, maintaining a balanced trade-off between precision and recall.

Importantly, the model exhibits stable behavior across different threshold settings, further supporting its robustness.

### Final Evaluation Considerations

The model selection is based on performance measured on a hold-out test set that was not used during training or hyperparameter optimization. While intermediate evaluations were performed during development, the test set remained independent of any model selection decisions, ensuring an unbiased assessment.

Although an additional validation on a completely unseen dataset would be ideal, the current evaluation framework already follows standard best practices by using a dedicated hold-out test set. Therefore, the reported performance provides a reliable estimate of real-world behavior.

### Conclusion

The histogram-based gradient boosting model offers the best overall balance between ranking performance and classification effectiveness. Given its superior ability to prioritize high-risk customers and its stable performance across evaluation scenarios, it represents the most suitable choice for deployment in a churn prediction setting.

## Executive Summary

This project developed and evaluated a range of machine learning models to predict customer churn and support targeted retention strategies. The primary objective was not only to accurately classify churn outcomes, but also to effectively **rank customers by their likelihood of churn**, enabling prioritization of intervention efforts.

Among all evaluated models, the **Histogram-based Gradient Boosting model** demonstrated the strongest overall performance. It achieved the highest ranking capability, as measured by ROC-AUC and PR-AUC, while maintaining competitive classification performance under a unified decision threshold. This makes it particularly well-suited for identifying high-risk customers and supporting data-driven decision-making.

An important finding of the analysis is that **threshold selection plays a critical role in model performance**, often influencing results as much as the choice of model itself. By adopting a global threshold, the system can maintain consistent and interpretable decision rules while preserving strong predictive performance.

While the selected model is best suited for ranking customers by risk, **interpretability remains essential for actionable insights**. In this regard, the logistic regression model provides valuable complementary information. Its coefficients offer direct insight into the factors associated with increased churn risk, allowing the business to better understand *why* customers are likely to churn.

This combination of models enables a two-step strategy:
1. **Identify high-risk customers** using the gradient boosting model  
2. **Inform retention actions** using insights derived from the logistic regression model  

Together, this approach supports both accurate prediction and meaningful intervention, aligning machine learning outputs with real-world business objectives.